## MAGIC Gamma Telescope Classification

The obective of this analysis is to investigate whether logistic regression and SVMs can distinguish between gamma-ray and hadron events using measurements collected from the MAGIC Gamma Telescope dataset provided [here](https://archive.ics.uci.edu/dataset/159/magic+gamma+telescope).

The target variable is `class`, which identifies whether the observed event is a gamma-ray event (`g`) or a hadron event (`h`) which makes the dataset suitable for binary classification.

In [ ]:
# Imports of highest import
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)
import time

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

RANDOM_STATE = 42

In [ ]:
columns = [
    "fLength",
    "fWidth",
    "fSize",
    "fConc",
    "fConc1",
    "fAsym",
    "fM3Long",
    "fM3Trans",
    "fAlpha",
    "fDist",
    "class"
]

df = pd.read_csv(
    "data/magic04.data",
    header=None,
    names=columns
)

print("Dataset dimensions:", df.shape)
print("\nNumber of observations:", df.shape[0])
print("Number of columns:", df.shape[1])
df.head()

In [ ]:
print(df["class"].value_counts())

print("\nClass proportions:")
print(df["class"].value_counts(normalize=True) * 100)
df.info()
df.isnull().sum()
df.describe().T

In [ ]:
features = df.drop(columns="class")
fig, axes = plt.subplots(5, 2, figsize=(15, 20))
for i, column in enumerate(features.columns):
    sns.histplot(
        data=df,
        x=column,
        kde=True,
        ax=axes[i // 2, i % 2]
    )
    axes[i // 2, i % 2].set_title(f"Distribution of {column}")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(15, 22))
for i, column in enumerate(features.columns):
    sns.boxplot(
        data=df,
        x="class",
        y=column,
        hue="class",
        legend=False,
        ax=axes[i // 2, i % 2]
    )
    axes[i // 2, i % 2].set_title(f"{column} by class")
plt.tight_layout()
plt.show()


In [ ]:
correlation_matrix = features.corr()
plt.figure(figsize=(12, 9))
sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap="coolwarm",
    center=0,
    fmt=".2f"
)
plt.title("Correlation Heatmap of Explanatory Variables")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.scatterplot(
    data=df,
    x="fLength",
    y="fWidth",
    hue="class",
    alpha=0.4,
    s=15,
    ax=axes[0]
)
axes[0].set_title("fLength vs fWidth")
sns.scatterplot(
    data=df,
    x="fAlpha",
    y="fDist",
    hue="class",
    alpha=0.4,
    s=15,
    ax=axes[1]
)
axes[1].set_title("fAlpha vs fDist")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 6))
sns.kdeplot(
    data=df,
    x="fAlpha",
    hue="class",
    fill=True,
    common_norm=False,
    alpha=0.3
)
plt.title("Distribution of fAlpha by Particle Class")
plt.xlabel("fAlpha")
plt.ylabel("Density")
plt.show()
df.groupby("class")[features.columns].median().T

### Observations

Chances are fAlpha will be our most promising indicator, and it should be noted that the strong correlation between some variables may indicate some overlapping information. The skewed distributions of fLength, fWidth, fSize, and fAlpha may also indicate some extreme observations to keep in mind. There's also a large class imbalance, so a classifier predicting gamma (`g`) for every observation would be 64.84% accurate. This is the minimum we need to beat!

In [ ]:
X = df.drop(columns="class")
y = df["class"]

# 80/20 test split, seed is 42
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)
print("Training observations:", len(X_train))
print("Testing observations:", len(X_test))
print("\nTraining class proportions:")
print(y_train.value_counts(normalize=True))
print("\nTesting class proportions:")
print(y_test.value_counts(normalize=True))

In [ ]:
logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=2000))
])

# Keep track of training runtime
start_time = time.perf_counter()
logistic_model.fit(X_train, y_train)
logistic_training_time = time.perf_counter() - start_time

# Generate predictions
y_pred = logistic_model.predict(X_test)

# Calculate prediction probabilities
y_prob = logistic_model.predict_proba(X_test)[:, 1]

# Evaluate performance
print("Logistic Regression Results")
print("---------------------------")

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, pos_label="g"))
print("Recall:", recall_score(y_test, y_pred, pos_label="g"))
print("F1-score:", f1_score(y_test, y_pred, pos_label="g"))
print("ROC-AUC:", roc_auc_score(y_test == "g", y_prob))
gamma_index = list(logistic_model.classes_).index("g")
y_prob = logistic_model.predict_proba(X_test)[:, gamma_index]
# Calculate ROC-AUC using gamma as the positive class
auc = roc_auc_score(
    (y_test == "g").astype(int),
    y_prob
)
print("Corrected ROC-AUC:", auc)

print("Training time:", logistic_training_time, "seconds")

In [ ]:
ConfusionMatrixDisplay.from_estimator(
    logistic_model,
    X_test,
    y_test,
    cmap="Blues"
)

plt.title("Logistic Regression Confusion Matrix")
plt.show()

In [ ]:
# Extract logistic regression coefficients
coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": logistic_model.named_steps["classifier"].coef_[0]
})

# Sort by absolute coefficient magnitude
coefficients["Absolute Coefficient"] = (
    coefficients["Coefficient"].abs()
)
coefficients = coefficients.sort_values(
    "Absolute Coefficient",
    ascending=False
)
print(coefficients)

plt.figure(figsize=(10, 6))
sns.barplot(
    data=coefficients,
    x="Coefficient",
    y="Feature",
    hue="Coefficient",
    palette="coolwarm",
    legend=False
)
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Logistic Regression Feature Coefficients")
plt.xlabel("Standardized Regression Coefficient")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

In [ ]:
print(classification_report(y_test, y_pred))
RocCurveDisplay.from_predictions(
    (y_test == "g").astype(int),
    y_prob
)

plt.title("Logistic Regression ROC Curve")
plt.plot([0, 1], [0, 1], linestyle="--", color="grey")
plt.show()